In [30]:
import warnings
warnings.filterwarnings("ignore", category=Warning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
!pip install tensorflow
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 3.7/351.2 MB 21.2 MB/s eta 0:00:17
    --------------------------------------- 8.4/351.2 MB 21.8 MB/s eta 0:00:16
   - -------------------------------------- 13.1/351.2 MB 21.7 MB/s eta 0:00:16
   -- ------------------------------------- 18.1/351.2 MB 22.0 MB/s eta 0:00:16
   -- ------------------------------------- 22.5/351.2 MB 22.0 MB/s eta 0:00:15
   --- ------------------------------------ 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
dftrain = pd.read_csv("data/train.csv")
dftrain.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [36]:
dftest = pd.read_csv("data/test.csv")
dftest.head()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
#Split data 
X = dftrain.drop("label", axis=1).values
y = dftrain["label"].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [38]:
#Reshape 
X_train = X_train.reshape(-1, 28, 28, 1)
X_val = X_val.reshape(-1, 28, 28, 1)

In [39]:
#Normalize 
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0

In [40]:
#Build CNN
num_conv1 = 32 
num_conv2 = 64 
num_hidden3 = 128  
num_output = 10
k_size = 3
p_size = 2


model = Sequential([
    Conv2D(num_conv1, kernel_size=(k_size, k_size),activation='relu', input_shape=(28, 28, 1)),

    MaxPooling2D(pool_size=(p_size, p_size)),

    Conv2D(num_conv2, kernel_size=(k_size, k_size), activation='relu'),
    MaxPooling2D(pool_size=(p_size, p_size)),

    Flatten(),

    Dense(num_hidden3, activation='relu'),
    Dense(num_output, activation='softmax') 
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)               │ (None, 26, 26, 32)    │          320 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 13, 13, 32)    │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_1 (Conv2D)             │ (None, 11, 11, 64)    │       18,496 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling2d_1               │ (None, 5, 5, 64)      │            0 │
│ (MaxPooling2D)                │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ flatten (Flatten)             │ (None, 1600)          │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 128)           │      204,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 10)            │        1,290 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [43]:
num_epochs = 10
history = model.fit(X_train, y_train, epochs=num_epochs, validation_split=0.2)

Epoch 1/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.9382 - loss: 0.2033 - val_accuracy: 0.9762 - val_loss: 0.0812
Epoch 2/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.9816 - loss: 0.0592 - val_accuracy: 0.9804 - val_loss: 0.0686
Epoch 3/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.9871 - loss: 0.0406 - val_accuracy: 0.9839 - val_loss: 0.0516
Epoch 4/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.9916 - loss: 0.0269 - val_accuracy: 0.9826 - val_loss: 0.0604
Epoch 5/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.9933 - loss: 0.0212 - val_accuracy: 0.9812 - val_loss: 0.0727
Epoch 6/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.9944 - loss: 0.0166 - val_accuracy: 0.9817 - val_loss: 0.0605
Epoch 7/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.9954 - loss: 0.0138 - val_accuracy: 0.9878 - val_loss: 0.0551
Epoch 8/10
840/840 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.9963 - loss: 0.0106 - 

In [44]:
#Evaluation
test_loss, test_accuracy = model.evaluate(X_val, y_val)
print(f"Validation accuracy: {test_accuracy}")

263/263 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9863 - loss: 0.0547
Validation accuracy: 0.9863095283508301
